# 🧪 Lab 2 — Linear Filtering & Template Matching

| | |
|---|---|
| **Sessions** | 6 – 8 (Linear filtering · Correlation & convolution · Gaussian · Template matching) |
| **Week** | 3 |
| **Learning Outcome** | **LO 2.1** Linear filtering & template matching |
| **Type** | Practice assignment — do it **before** your Lab workshop (Session 8) |
| **Estimated time** | 90 – 120 minutes |

---

### 🎯 Objectives
1. Implement **2-D cross-correlation** and **convolution** from scratch and relate them to OpenCV.
2. Use **box / Gaussian / median** filtering to clean noisy images and compare them quantitatively (MSE).
3. Explain the difference between correlation and convolution, and why the median filter beats the mean filter on salt-and-pepper noise.
4. Implement **SSD** and **NCC** template matching and locate a known template inside an image.

> 💡 Fill in every `# ==== YOUR CODE HERE ====` block and answer the markdown
> questions. Keep outputs visible and **Run All** before submitting.



## Setup



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)
print("✅ numpy", np.__version__, "| opencv", cv2.__version__)



## Part 1 · Correlation and convolution from scratch

**Recap.** Filtering replaces each pixel by a weighted sum of its neighbours. In
**cross-correlation** the kernel is *not* flipped:

$$(I \star K)[i,j] = \sum_{m,n} K[m,n]\; I[i+m,\; j+n]$$

In **convolution** the kernel is flipped both ways:

$$(I * K)[i,j] = \sum_{m,n} K[m,n]\; I[i-m,\; j-n]$$

For a **symmetric** kernel (box, Gaussian) the two are identical; for an
**asymmetric** kernel they differ. Boundaries are usually handled by padding
(zero, reflect, clamp…).



### 🛠️ Task 1 — 2-D cross-correlation 
Complete `correlate2d` below. Use **zero padding** so the output has the same size
as the input. *Hint: `np.pad(img, pad, mode='constant')` gives you a padded copy
and then you loop over valid positions, taking `kernel.shape`-sized patches.*



In [ ]:
def correlate2d(img, kernel):
    """Cross-correlate img (2-D) with kernel using ZERO padding."""
    kh, kw = kernel.shape; ph, pw = kh // 2, kw // 2
    padded = np.pad(img, ((ph, ph), (pw, pw)), mode='constant')
    out = np.zeros_like(img, dtype=np.float64)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            out[i, j] = np.sum(padded[i:i+kh, j:j+kw] * kernel)
    return out

img = np.arange(36, dtype=np.float64).reshape(6, 6)
box = np.ones((3, 3), dtype=np.float64)
mine = correlate2d(img, box)
ref = cv2.filter2D(img, cv2.CV_64F, box, borderType=cv2.BORDER_CONSTANT)
print("max |mine - cv2.filter2D| =", np.abs(mine - ref).max())


In [ ]:
# ✅ Self check — do not edit
img = np.arange(36, dtype=np.float64).reshape(6, 6)
box = np.ones((3, 3), dtype=np.float64)
ref = cv2.filter2D(img, cv2.CV_64F, box, borderType=cv2.BORDER_CONSTANT)
assert np.allclose(correlate2d(img, box), ref, atol=1e-8), "correlate2d does not match cv2.filter2D"
print("✅ Task 1 assertions passed!")



### 🛠️ Task 2 — Convolution vs correlation 
`convolve2d` is just `correlate2d` with the kernel **flipped**. Complete it, then
correlate and convolve the *same* asymmetric kernel with an image and look at the
difference.



In [ ]:
def convolve2d(img, kernel):
    """Convolution = correlation with the kernel flipped both ways."""
    return correlate2d(img, np.flip(kernel))

k_asym = np.array([[0., 0., 0.], [0., -1., 1.], [0., 0., 0.]])
edge = np.zeros((60, 90)); edge[:, 45:] = 1.0
corr = correlate2d(edge, k_asym); conv = convolve2d(edge, k_asym)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(edge, cmap='gray'); ax[0].set_title("step edge"); ax[0].axis('off')
ax[1].imshow(corr, cmap='RdBu_r'); ax[1].set_title("correlation"); ax[1].axis('off')
ax[2].imshow(conv, cmap='RdBu_r'); ax[2].set_title("convolution"); ax[2].axis('off')
plt.tight_layout(); plt.show()


### 📝 Task 3 — Why do we flip?
1. Correlation has its positive peak on the left side of this rising step; convolution has it on the right.
2. A shift-invariant imaging system combines each input location with an impulse response centred relative to the output, which is mathematically convolution.
3. The distinction disappears for kernels symmetric under a 180° rotation, such as Gaussian and box kernels.


## Part 2 · Removing noise: box, Gaussian, median

**Recap.** Sensors add noise. Two common models are **Gaussian noise** (small
random perturbations on every pixel) and **impulse / salt-and-pepper noise**
(random pixels turned fully black or white). Linear filters (box, Gaussian)
suppress Gaussian noise but smear edges; the **median filter** is a non-linear
filter that removes salt-and-pepper almost perfectly while preserving edges.



In [ ]:
# ---- provided: build a clean image and two noisy versions -----------------
H, W = 160, 220
clean = np.zeros((H, W), dtype=np.float64)
cv2.rectangle(clean, (30, 30), (130, 130), 1.0, -1)          # white square
cv2.circle(clean, (180, 100), 30, 0.6, -1)                   # grey circle
cv2.line(clean, (10, 150), (210, 40), 0.9, 4)                # bright diagonal line

gauss = clean + np.random.normal(0, 0.12, (H, W))            # Gaussian noise
gauss = np.clip(gauss, 0, 1)
sp = clean.copy()
n_pepper = int(0.03 * H * W)                                  # 3% salt & pepper
idx = np.random.choice(H * W, n_pepper, replace=False)
sp.ravel()[idx] = 0.0
idx = np.random.choice(H * W, n_pepper, replace=False)
sp.ravel()[idx] = 1.0

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(clean, cmap='gray'); ax[0].set_title("clean"); ax[0].axis('off')
ax[1].imshow(gauss, cmap='gray'); ax[1].set_title("+ Gaussian noise"); ax[1].axis('off')
ax[2].imshow(sp, cmap='gray');    ax[2].set_title("+ salt & pepper"); ax[2].axis('off')
plt.tight_layout(); plt.show()



### 🛠️ Task 4 — A proper Gaussian kernel 
Implement `gaussian_kernel(size, sigma)`: build the 1-D vector
$g(x)=e^{-x^2/(2\sigma^2)}$ for $x=-(k-1)/2\ldots(k-1)/2$, take the outer product
$K=g\otimes g$ and **normalise** it so that $\sum K = 1$.



In [ ]:
def gaussian_kernel(size, sigma):
    """Return a normalised (size x size) Gaussian kernel."""
    if size <= 0 or size % 2 == 0 or sigma <= 0:
        raise ValueError("size must be positive and odd; sigma must be positive")
    coords = np.arange(size, dtype=np.float64) - size // 2
    xx, yy = np.meshgrid(coords, coords)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel / kernel.sum()

k5 = gaussian_kernel(5, 1.0)
print("kernel sum =", k5.sum(), "| shape", k5.shape)
plt.imshow(k5, cmap='viridis'); plt.title("5x5 Gaussian, sigma=1"); plt.colorbar(); plt.show()
smooth_gauss = cv2.filter2D(gauss, cv2.CV_64F, k5, borderType=cv2.BORDER_REFLECT)
smooth_sp = cv2.filter2D(sp, cv2.CV_64F, k5, borderType=cv2.BORDER_REFLECT)
def mse(a, b): return float(np.mean((a - b) ** 2))
print("Gaussian smoothing of Gaussian noise:  MSE %.5f" % mse(smooth_gauss, clean))
print("Gaussian smoothing of salt & pepper:   MSE %.5f" % mse(smooth_sp, clean))


In [ ]:
# ✅ Self check — do not edit
g = gaussian_kernel(5, 1.0)
assert abs(g.sum() - 1.0) < 1e-9, "kernel must be normalised"
assert g.shape == (5, 5), "kernel must be size x size"
assert np.all(g >= 0), "Gaussian values are non-negative"
print("✅ Task 4 assertions passed!")



### 🛠️ Task 5 — Median beats the mean 
Apply a 5×5 **median filter** (`cv2.medianBlur`, ksize=5) to *both* noisy images,
and also a 5×5 **box (mean)** filter (`cv2.boxFilter` or your `correlate2d` with a
normalised box). Compute MSE against `clean` for all four combinations and fill the
table below.



In [ ]:
med_gauss = cv2.medianBlur(np.uint8(gauss * 255), 5).astype(np.float64) / 255.0
med_sp = cv2.medianBlur(np.uint8(sp * 255), 5).astype(np.float64) / 255.0

box = np.ones((5, 5)) / 25.0
box_gauss = cv2.filter2D(gauss, cv2.CV_64F, box, borderType=cv2.BORDER_REFLECT)
box_sp = cv2.filter2D(sp, cv2.CV_64F, box, borderType=cv2.BORDER_REFLECT)

print("                | Gaussian noise | salt & pepper |")
print("box (mean) 5x5 |  %.5f       |   %.5f    |" % (mse(box_gauss, clean), mse(box_sp, clean)))
print("median 5x5     |  %.5f       |   %.5f    |" % (mse(med_gauss, clean), mse(med_sp, clean)))



### 📝 Task 6 — Interpret the table
1. Gaussian filtering is generally best for additive Gaussian noise because it averages many independent perturbations while weighting nearby pixels smoothly.
2. The median filter is best for salt-and-pepper noise: it rejects isolated extreme black/white outliers instead of averaging them into their neighbours.
3. A large Gaussian suppresses more noise but also blurs edges and removes fine detail.


## Part 3 · Template matching

**Recap.** Given a (small) template $T$ and an image $I$, template matching slides
$T$ over $I$ and scores each position with a similarity/distance measure. Two
classical scores are

$$\mathrm{SSD}(i,j)=\sum_{m,n}\big(I(i+m,j+n)-T(m,n)\big)^2 \qquad\text{(small = good)}$$

$$\mathrm{NCC}(i,j)=\frac{\sum (I-\bar I_w)(T-\bar T)}
{\sqrt{\sum (I-\bar I_w)^2}\sqrt{\sum (T-\bar T)^2}} \qquad\text{(large = good)}$$

NCC is invariant to additive brightness and multiplicative contrast; SSD is not.



In [ ]:
# ---- provided: hide a template inside an image at a KNOWN place -------------
base = np.random.default_rng(7).integers(0, 255, (180, 240)).astype(np.float64)  # random 'texture'
templ = base[60:90, 100:130].copy()              # take a true patch from the texture
img_with = base.copy()
# paste the template (brightened + noisy) at a known location (row 40, col 80)
patch = np.clip(templ + 40 + np.random.default_rng(1).normal(0, 8, templ.shape), 0, 255)
img_with[40:70, 80:110] = patch

TRUE_ROW, TRUE_COL = 40, 80
print("Ground-truth top-left of template:", (TRUE_ROW, TRUE_COL))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(img_with, cmap='gray'); ax[0].set_title("image with hidden template"); ax[0].axis('off')
ax[1].imshow(patch, cmap='gray');     ax[1].set_title("the template we search for"); ax[1].axis('off')
plt.tight_layout(); plt.show()



### 🛠️ Task 7 — SSD template matching 
Complete `match_ssd`, which slides the template and returns the **SSD map** (same
shape as the image, with `np.inf` in the border region where the template does not
fit). Then find the minimum and report the detected location.



In [ ]:
def match_ssd(img, templ):
    """Return the SSD map. Border positions where the template does not fit -> inf."""
    H, W = img.shape; th, tw = templ.shape
    ssd = np.full((H, W), np.inf)
    templ = templ.astype(np.float64)
    for i in range(H - th + 1):
        for j in range(W - tw + 1):
            diff = img[i:i+th, j:j+tw].astype(np.float64) - templ
            ssd[i, j] = np.sum(diff**2)
    return ssd

ssd_map = match_ssd(img_with, patch)
r, c = np.unravel_index(np.argmin(ssd_map), ssd_map.shape)
print("Detected top-left by SSD  :", (r, c), " | truth:", (TRUE_ROW, TRUE_COL))
plt.imshow(ssd_map, cmap='viridis'); plt.title("SSD map (darker = better match)"); plt.colorbar(); plt.show()


In [ ]:
# ✅ Self check — do not edit
ssd_map = match_ssd(img_with, patch)
r, c = np.unravel_index(np.argmin(ssd_map), ssd_map.shape)
assert abs(r - TRUE_ROW) <= 2 and abs(c - TRUE_COL) <= 2, \
    "SSD minimum is not at the hidden template location"
print("✅ Task 7 assertions passed — SSD located the template!")



### 🛠️ Task 8 — NCC and brightness invariance
Complete `match_ncc` (use the formula above with local window means/std), and
compare with OpenCV's `cv2.matchTemplate(..., TM_CCOEFF_NORMED)`. Then answer the
conceptual question.



In [ ]:
def match_ncc(img, templ):
    """Return the NCC map (higher = better). Non-fitting borders -> -inf."""
    H, W = img.shape; th, tw = templ.shape
    ncc = np.full((H, W), -np.inf)
    t_bar = templ.astype(np.float64) - np.mean(templ)
    t_norm = np.linalg.norm(t_bar)
    for i in range(H - th + 1):
        for j in range(W - tw + 1):
            p_bar = img[i:i+th, j:j+tw].astype(np.float64); p_bar -= p_bar.mean()
            den = np.linalg.norm(p_bar) * t_norm
            ncc[i, j] = np.sum(p_bar * t_bar) / den if den > 0 else 0.0
    return ncc

ncc_map = match_ncc(img_with, patch)
r2, c2 = np.unravel_index(np.argmax(ncc_map), ncc_map.shape)
print("Detected top-left by NCC  :", (r2, c2), " | truth:", (TRUE_ROW, TRUE_COL))
res = cv2.matchTemplate(img_with.astype(np.float32), patch.astype(np.float32), cv2.TM_CCOEFF_NORMED)
r3, c3 = np.unravel_index(np.argmax(res), res.shape)
print("Detected top-left by cv2  :", (r3, c3))


In [ ]:
# ✅ Self check — do not edit
ncc_map = match_ncc(img_with, patch)
r2, c2 = np.unravel_index(np.argmax(ncc_map), ncc_map.shape)
assert abs(r2 - TRUE_ROW) <= 2 and abs(c2 - TRUE_COL) <= 2, "NCC peak is not at the template"
res = cv2.matchTemplate(img_with.astype(np.float32), patch.astype(np.float32), cv2.TM_CCOEFF_NORMED)
r3, c3 = np.unravel_index(np.argmax(res), res.shape)
assert abs(r3 - TRUE_ROW) <= 2 and abs(c3 - TRUE_COL) <= 2, "cv2 NCC should find it too"
print("✅ Task 8 assertions passed — NCC (yours and OpenCV's) found the template!")



### 📝 Task 9 — Why NCC?
1. Not reliably: SSD directly penalises the +40 intensity offset, so its minimum can move to a different region with a closer absolute brightness.
2. It can fail under strong scale, rotation, perspective, or occlusion changes; feature matching is more robust in those cases.


## ✅ Lab 2 — Reflection
NCC is clearest after this lab; a useful workshop question is how to choose a template size that balances specificity against tolerance to deformation.
